In [51]:
# Cài đặt thư viện OpenAI API (chạy 1 lần đầu)
!pip install -q openai

import abc
import json
import os
from typing import Dict, List, Any, Optional
from openai import OpenAI
from langchain.agents import create_agent
from langchain_deepseek.chat_models import ChatDeepSeek

from dotenv import load_dotenv

load_dotenv()

True

In [52]:
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if not DEEPSEEK_API_KEY:
    raise ValueError("Chưa tìm thấy DEEPSEEK_API_KEY trong file .env")

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

print("Đã khởi tạo DeepSeek Client!")

Đã khởi tạo DeepSeek Client!


In [53]:
class AbstractPersistence(abc.ABC):
    """Interface định nghĩa hợp đồng lưu trữ cho ứng dụng."""
    @abc.abstractmethod
    def save_state(self, session_id: str, data: Dict[str, Any]) -> None:
        pass

    @abc.abstractmethod
    def load_state(self, session_id: str) -> Optional[Dict[str, Any]]:
        pass

    @abc.abstractmethod
    def append_log(self, session_id: str, log_entry: Dict[str, Any]) -> None:
        pass

In [54]:
class InMemoryPersistence(AbstractPersistence):
    """Lớp lưu trữ trạng thái trên RAM."""
    def __init__(self):
        self._states: Dict[str, Dict[str, Any]] = {}
        self._logs: Dict[str, List[Dict[str, Any]]] = {}

    def save_state(self, session_id: str, data: Dict[str, Any]) -> None:
        self._states[session_id] = data

    def load_state(self, session_id: str) -> Optional[Dict[str, Any]]:
        return self._states.get(session_id)

    def append_log(self, session_id: str, log_entry: Dict[str, Any]) -> None:
        if session_id not in self._logs:
            self._logs[session_id] = []
        self._logs[session_id].append(log_entry)

In [55]:
class FilePersistence(AbstractPersistence):
    """Lớp lưu trữ trạng thái dưới dạng File JSON trong File System."""
    def __init__(self, storage_dir: str = "./agent_logs"):
        self.storage_dir = storage_dir
        os.makedirs(self.storage_dir, exist_ok=True)

    def _get_path(self, session_id: str) -> str:
        return os.path.join(self.storage_dir, f"{session_id}.json")

    def save_state(self, session_id: str, data: Dict[str, Any]) -> None:
        file_path = self._get_path(session_id)
        current_data = {"state": {}, "logs": []}
        if os.path.exists(file_path):
            with open(file_path, "r", encoding="utf-8") as f:
                current_data = json.load(f)

        current_data["state"] = data
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(current_data, f, indent=2, ensure_ascii=False)

    def load_state(self, session_id: str) -> Optional[Dict[str, Any]]:
        file_path = self._get_path(session_id)
        if not os.path.exists(file_path):
            return None
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f).get("state")

    def append_log(self, session_id: str, log_entry: Dict[str, Any]) -> None:
        file_path = self._get_path(session_id)
        current_data = {"state": {}, "logs": []}
        if os.path.exists(file_path):
            with open(file_path, "r", encoding="utf-8") as f:
                current_data = json.load(f)

        current_data.setdefault("logs", []).append(log_entry)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(current_data, f, indent=2, ensure_ascii=False)

In [56]:
print("Persistence Layer đã được khởi tạo sẵn sàng!")

Persistence Layer đã được khởi tạo sẵn sàng!


In [57]:
class BaseAgent(abc.ABC):
    def __init__(self, name: str, role: str, llm_client: OpenAI):
        self.name = name
        self.role = role
        self.client = llm_client

    @abc.abstractmethod
    def execute(self, task_input: str, context: Optional[str] = None) -> str:
        pass

In [58]:
class CoderAgent(BaseAgent):
    """Agent chịu trách nhiệm sinh mã nguồn Python."""
    def execute(self, task_input: str, context: Optional[str] = None) -> str:
        system_prompt = (
            "You are an expert Python Software Engineer. "
            "Write clean, modular, and efficient Python code with docstrings and type hints. "
            "Return ONLY the executable Python code inside a markdown code block."
        )
        
        user_prompt = f"Task: {task_input}"
        if context:
            user_prompt += f"\n\nPrevious Feedback / Errors to fix:\n{context}"

        response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content.strip()

In [59]:
class ReviewerAgent(BaseAgent):
    """Agent chịu trách nhiệm đánh giá an toàn, chất lượng mã nguồn."""
    def execute(self, code: str, context: Optional[str] = None) -> str:
        system_prompt = (
            "You are a Senior Code Auditor and Security Expert. "
            "Analyze the provided code for correctness, type hints, docstrings, edge cases, and security vulnerabilities. "
            "If the code meets high standards, start your response strictly with 'PASS'. "
            "If there are issues, start your response strictly with 'REJECT' followed by a detailed list of required improvements."
        )

        response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Code to review:\n{code}"}
            ],
            temperature=0.1
        )
        return response.choices[0].message.content.strip()

In [60]:
class DebuggerAgent(BaseAgent):
    """Agent sửa code dựa trên phản hồi của Reviewer."""
    def execute(self, code: str, context: Optional[str] = None) -> str:
        system_prompt = (
            "You are a Specialized Debugging and Refactoring Expert. "
            "Your task is to take buggy or rejected code along with the Reviewer's feedback, "
            "fix all reported issues, and return the corrected, functional Python code inside a markdown code block."
        )

        user_prompt = f"Buggy Code:\n{code}\n\nReviewer Feedback / Issues:\n{context}"

        response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1
        )
        return response.choices[0].message.content.strip()

In [61]:
print("Các Agents đã được định nghĩa!")

Các Agents đã được định nghĩa!


In [ ]:
class MultiAgentOrchestrator:
    def __init__(self, persistence: AbstractPersistence, llm_client: OpenAI, max_iterations: int = 3):
        if max_iterations < 1:
            raise ValueError("max_iterations phải lớn hơn hoặc bằng 1")

        self.persistence = persistence
        self.max_iterations = max_iterations
        self.coder = CoderAgent("Coder", "Developer", llm_client)
        self.reviewer = ReviewerAgent("Reviewer", "Security Auditor", llm_client)
        self.debugger = DebuggerAgent("Debugger", "Debugging Expert", llm_client)

    def _call_agent(
        self,
        session_id: str,
        agent: BaseAgent,
        request: str,
        context: Optional[str],
        iteration: int,
    ) -> Optional[str]:
        interaction = {
            "iteration": iteration,
            "agent": agent.name,
            "role": agent.role,
            "request": request,
            "context": context,
        }

        try:
            response = agent.execute(request, context=context)
            interaction.update({
                "status": "SUCCESS",
                "response": response,
                "error": None,
            })
            return response
        except Exception as error:
            interaction.update({
                "status": "ERROR",
                "response": None,
                "error": {
                    "type": type(error).__name__,
                    "message": str(error),
                },
            })
            raise
        finally:
            self.persistence.append_log(session_id, interaction)

    def run_task(self, session_id: str, task_description: str) -> Dict[str, Any]:
        current_code = ""
        status = "IN_PROGRESS"
        iteration = 0

        try:
            current_code = self._call_agent(
                session_id=session_id,
                agent=self.coder,
                request=task_description,
                context=None,
                iteration=0,
            ) or ""

            for iteration in range(1, self.max_iterations + 1):
                print(f"\n[DINHHUONG] --- Iteration {iteration}/{self.max_iterations} ---")
                print("### Reviewer Agent is auditing code...")
                review_result = self._call_agent(
                    session_id=session_id,
                    agent=self.reviewer,
                    request=current_code,
                    context=None,
                    iteration=iteration,
                ) or ""

                if review_result.startswith("PASS"):
                    print("[DINHHUONG] Code APPROVED by Reviewer Agent!")
                    status = "COMPLETED"
                    break

                print("[DINHHUONG] Code REJECTED! Feedback sent to Debugger Agent.")
                if iteration < self.max_iterations:
                    current_code = self._call_agent(
                        session_id=session_id,
                        agent=self.debugger,
                        request=current_code,
                        context=review_result,
                        iteration=iteration,
                    ) or ""

            final_state = {
                "session_id": session_id,
                "status": status,
                "final_code": current_code,
                "total_iterations": iteration,
            }
        except Exception as error:
            status = "FAILED"
            final_state = {
                "session_id": session_id,
                "status": status,
                "final_code": current_code,
                "total_iterations": iteration,
                "error": {
                    "type": type(error).__name__,
                    "message": str(error),
                },
            }
            print(f"❌ Workflow failed: {type(error).__name__}: {error}")

        self.persistence.save_state(session_id, final_state)
        return final_state

print("Orchestrator is ready!")

✅ Orchestrator sẵn sàng!


In [63]:
# Khởi tạo lưu trữ File System
file_storage = FilePersistence(storage_dir="./agent_logs")

orchestrator = MultiAgentOrchestrator(
    persistence=file_storage,
    llm_client=client,
    max_iterations=3
)

# Chạy bài toán lập trình thực tế
task = "Viết hàm Python đọc file JSON và trích xuất danh sách các email hợp lệ bằng Regex."
result = orchestrator.run_task(session_id="task_deepseek_02", task_description=task)

print("\n" + "="*50)
print("### RESULTS SUMMARY")
print("="*50)
print("Session ID:", result["session_id"])
print("Status:", result["status"])
print("Total Iterations:", result["total_iterations"])
print("\n### FINAL GENERATED CODE:")
print(result["final_code"])


🔄 --- Iteration 1/3 ---
### Reviewer Agent is auditing code...
✅ Code APPROVED by Reviewer Agent!

### RESULTS SUMMARY
Session ID: task_deepseek_02
Status: COMPLETED
Total Iterations: 1

### FINAL GENERATED CODE:
```python
import json
import re
from typing import List, Any


def extract_valid_emails(json_file_path: str) -> List[str]:
    """
    Đọc file JSON và trích xuất danh sách các email hợp lệ bằng Regex.

    Hàm sẽ duyệt đệ quy toàn bộ cấu trúc dữ liệu trong file JSON
    (dict, list, str) để tìm tất cả các chuỗi khớp với mẫu email hợp lệ.

    Args:
        json_file_path (str): Đường dẫn tới file JSON cần đọc.

    Returns:
        List[str]: Danh sách các email hợp lệ (đã loại bỏ trùng lặp,
                   giữ nguyên thứ tự xuất hiện).

    Raises:
        FileNotFoundError: Nếu file không tồn tại.
        json.JSONDecodeError: Nếu nội dung file không phải JSON hợp lệ.
    """
    # Regex kiểm tra email hợp lệ (chuẩn thực dụng, không quá khắt khe)
    email_pattern = re.

In [64]:
log_file_path = "./agent_logs/task_deepseek_02.json"
if os.path.exists(log_file_path):
    with open(log_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("Log file storage path:", log_file_path)
    print(json.dumps(data, indent=2, ensure_ascii=False)[:1000] + "\n... [Truncated]")

Log file storage path: ./agent_logs/task_deepseek_02.json
{
  "state": {
    "session_id": "task_deepseek_02",
    "status": "COMPLETED",
    "final_code": "```python\nimport json\nimport re\nfrom typing import List, Any\n\n\ndef extract_valid_emails(json_file_path: str) -> List[str]:\n    \"\"\"\n    Đọc file JSON và trích xuất danh sách các email hợp lệ bằng Regex.\n\n    Hàm sẽ duyệt đệ quy toàn bộ cấu trúc dữ liệu trong file JSON\n    (dict, list, str) để tìm tất cả các chuỗi khớp với mẫu email hợp lệ.\n\n    Args:\n        json_file_path (str): Đường dẫn tới file JSON cần đọc.\n\n    Returns:\n        List[str]: Danh sách các email hợp lệ (đã loại bỏ trùng lặp,\n                   giữ nguyên thứ tự xuất hiện).\n\n    Raises:\n        FileNotFoundError: Nếu file không tồn tại.\n        json.JSONDecodeError: Nếu nội dung file không phải JSON hợp lệ.\n    \"\"\"\n    # Regex kiểm tra email hợp lệ (chuẩn thực dụng, không quá khắt khe)\n    email_pattern = re.compile(\n        r\"[a-zA

## Kịch bản kiểm thử

Test giả lập luồng Coder, Reviewer và Debugger mà không gọi API thật.

In [65]:
class FakeMessage:
    def __init__(self, content: str):
        self.content = content


class FakeChoice:
    def __init__(self, content: str):
        self.message = FakeMessage(content)


class FakeCompletions:
    def __init__(self, responses: List[str]):
        self.responses = iter(responses)
        self.calls: List[Dict[str, Any]] = []

    def create(self, **kwargs: Any) -> Any:
        self.calls.append(kwargs)
        return type("FakeResponse", (), {
            "choices": [FakeChoice(next(self.responses))]
        })()


class FakeClient:
    def __init__(self, responses: List[str]):
        self.chat = type("FakeChat", (), {
            "completions": FakeCompletions(responses)
        })()


class ErrorCompletions:
    def create(self, **kwargs: Any) -> Any:
        raise RuntimeError("DeepSeek unavailable")


class ErrorClient:
    def __init__(self):
        self.chat = type("FakeChat", (), {
            "completions": ErrorCompletions()
        })()


def test_three_agent_workflow() -> None:
    responses = [
        "def broken_function():\n    return missing_name",
        "REJECT: missing_name is undefined",
        "def fixed_function():\n    return 42",
        "PASS: function is valid"
    ]
    fake_client = FakeClient(responses)
    persistence = InMemoryPersistence()
    test_orchestrator = MultiAgentOrchestrator(
        persistence=persistence,
        llm_client=fake_client,
        max_iterations=2
    )

    state = test_orchestrator.run_task(
        session_id="test_three_agent_workflow",
        task_description="Create a function that returns 42."
    )
    logs = persistence._logs["test_three_agent_workflow"]
    agents = [entry["agent"] for entry in logs]

    assert state["status"] == "COMPLETED"
    assert state["final_code"] == "def fixed_function():\n    return 42"
    assert agents == ["Coder", "Reviewer", "Debugger", "Reviewer"]
    assert [entry["status"] for entry in logs] == ["SUCCESS"] * 4
    assert logs[0]["request"] == "Create a function that returns 42."
    assert logs[1]["request"] == "def broken_function():\n    return missing_name"
    assert logs[2]["context"] == "REJECT: missing_name is undefined"
    assert len(fake_client.chat.completions.calls) == 4


def test_coder_error_is_logged() -> None:
    persistence = InMemoryPersistence()
    test_orchestrator = MultiAgentOrchestrator(
        persistence=persistence,
        llm_client=ErrorClient(),
        max_iterations=1
    )

    state = test_orchestrator.run_task(
        session_id="test_coder_error",
        task_description="Generate code."
    )
    log_entry = persistence._logs["test_coder_error"][0]

    assert state["status"] == "FAILED"
    assert log_entry["agent"] == "Coder"
    assert log_entry["status"] == "ERROR"
    assert log_entry["response"] is None
    assert log_entry["error"] == {
        "type": "RuntimeError",
        "message": "DeepSeek unavailable",
    }


def test_persistence_round_trip() -> None:
    persistence = InMemoryPersistence()
    expected = {"status": "COMPLETED", "total_iterations": 1}
    persistence.save_state("persistence_test", expected)
    persistence.append_log("persistence_test", {"agent": "Tester"})

    assert persistence.load_state("persistence_test") == expected
    assert persistence._logs["persistence_test"] == [{"agent": "Tester"}]


def test_file_persistence_keeps_logs() -> None:
    import tempfile

    with tempfile.TemporaryDirectory() as storage_dir:
        persistence = FilePersistence(storage_dir=storage_dir)
        persistence.append_log("file_test", {"agent": "Reviewer"})
        persistence.save_state("file_test", {"status": "COMPLETED"})

        with open(os.path.join(storage_dir, "file_test.json"), "r", encoding="utf-8") as f:
            saved_data = json.load(f)

        assert saved_data["state"] == {"status": "COMPLETED"}
        assert saved_data["logs"] == [{"agent": "Reviewer"}]


def test_invalid_iteration_limit() -> None:
    try:
        MultiAgentOrchestrator(
            persistence=InMemoryPersistence(),
            llm_client=FakeClient([]),
            max_iterations=0
        )
    except ValueError:
        return
    raise AssertionError("max_iterations=0 phải tạo ValueError")


test_three_agent_workflow()
test_coder_error_is_logged()
test_persistence_round_trip()
test_file_persistence_keeps_logs()
test_invalid_iteration_limit()
print("Tất cả kịch bản kiểm thử đều PASS")


🔄 --- Iteration 1/2 ---
### Reviewer Agent is auditing code...
❌ Code REJECTED! Feedback sent to Debugger Agent.

🔄 --- Iteration 2/2 ---
### Reviewer Agent is auditing code...
✅ Code APPROVED by Reviewer Agent!
❌ Workflow failed: RuntimeError: DeepSeek unavailable
Tất cả kịch bản kiểm thử đều PASS
